# device-consistent-construct — faded example 1: Add a scratch bias buffer with the input's device + dtype

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `device-consistent-construct`. Running the beacon reports progress on the `PyTorch: Device-consistent tensor construction` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `PyTorch: Device-consistent tensor construction` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`device-consistent-construct`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "device-consistent-construct"
DD_SUBTOPIC = "PyTorch: Device-consistent tensor construction"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

A scratch tensor created inside `forward()` must thread `device=x.device` and `dtype=x.dtype` so it can be added to `x` without a device-mismatch error or a dtype upcast. The naive `t.full(shape, v).to(x.device)` defaults to CPU `float32` and breaks on non-float32 / non-CPU inputs.

## Faded exercise 1

### Faded — scaled bias buffer

Implement a Module whose `forward(x)` builds a constant-`0.5` buffer the same shape as `x`, then returns `x + buf`. The buffer MUST be allocated directly on `x`'s device and in `x`'s dtype so the addition never upcasts. Complete the blanked allocation line.

**Fill in:** Allocate a buffer of constant 0.5 with the same shape, device, and dtype as x.

In [ ]:
def make_bias_adder():
    class BiasAdder(t.nn.Module):
        def forward(self, x):
            buf = None  # TODO: allocate a buffer of constant 0.5 with the same shape, device, and dtype as x
            return x + buf
    return BiasAdder()


def _test():
    mod = make_bias_adder()
    for dt in (t.float32, t.float64, t.bfloat16):
        t.manual_seed(0)
        x = t.randn(3, 4).to(dt)
        y = mod(x)
        assert y.dtype == dt, (y.dtype, dt)
        assert y.device == x.device
        assert y.shape == x.shape
        expected = x + t.full(x.shape, 0.5, device=x.device, dtype=dt)
        assert t.allclose(y.float(), expected.float(), atol=1e-2)


try:
    _test()
    _dd_passed.add('faded1')
    print('[Delta Drills] faded1 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
def make_bias_adder():
    class BiasAdder(t.nn.Module):
        def forward(self, x):
            buf = t.full(x.shape, 0.5, device=x.device, dtype=x.dtype)
            return x + buf
    return BiasAdder()
```
</details>